# Transformers: a hands-on implementation

**Track:** Language models · **Guide:** [`033-transformers.md`](../guides/concepts/033-transformers.md)

Transformers use attention to let sequence positions combine information directly, with positional information added separately.

> **Intuition:** Each word can consult a relevance-weighted group discussion rather than passing a message down a long chain.

This lesson builds a small working example in stages: prepare inputs, implement
the core idea, run it on a realistic toy case, and inspect its output and limits.
It runs offline unless a cell is explicitly marked as an optional provider call.

```mermaid
flowchart LR
    A[Prepare inputs] --> B[Implement Transformers]
    B --> C[Run on example]
    C --> D[Evaluate and inspect]
```

**Case study:** Compare a long-range dependency handled by attention with the same dependency in a recurrent model.

Run the cells from top to bottom. To install the notebook environment, follow
[`guides/SETUP.md`](../guides/SETUP.md). The toy data and implementation are for
learning, not a production quality, safety, or security guarantee.


## 1. Prepare query, key, and value vectors

Self-attention compares a query with keys, then uses the resulting weights to mix values.


In [ ]:
import numpy as np

tokens = ["the", "charger", "smokes"]
Q = np.array([[1.0, 0.0], [0.0, 1.0], [0.9, 0.1]])
K = Q.copy()
V = np.array([[0.1, 0.0], [0.0, 1.0], [1.0, 0.2]])
print("token count:", len(tokens), "embedding width:", Q.shape[1])

## 2. Compute scaled dot-product attention

Softmax turns scaled compatibility scores into weights. Each row says how much one token reads from every other token.


In [ ]:
def softmax(values, axis=-1):
    shifted = values - values.max(axis=axis, keepdims=True)
    weights = np.exp(shifted)
    return weights / weights.sum(axis=axis, keepdims=True)

scores = Q @ K.T / np.sqrt(K.shape[1])
weights = softmax(scores)
context = weights @ V
print("attention weights:")
print(np.round(weights, 2))
print("context vectors:")
print(np.round(context, 2))

## 3. Apply a causal mask

A decoder-only language model must not look at future tokens when predicting the next token.


In [ ]:
causal_mask = np.triu(np.ones_like(scores, dtype=bool), k=1)
masked_scores = np.where(causal_mask, -1e9, scores)
causal_weights = softmax(masked_scores)
print("causal weights:")
print(np.round(causal_weights, 2))
assert np.allclose(causal_weights[0, 1:], 0)

## Wrap-up and next steps

- Identify what was learned, stored, retrieved, or generated in this implementation.
- Change an input and observe how the output changes.
- Test one failure case and decide when the system should abstain or ask for help.
- Replace toy data/components only after you can evaluate the change against a
  held-out or representative test set.

Continue with the [concept guide](../guides/concepts/033-transformers.md) and the
[learning roadmap](../guides/ROADMAP.md).
